In [1]:
from numba import cuda , float32
import numpy as np
import time

In [2]:
# Controls threads per block and shared memory usage.
# The computation will be done on blocks of TPBxTPB elements.
TPB = 16



@cuda.jit
def fast_matmul(A, B, C):
    
    # Define an array in the shared memory
    # The size and type of the arrays must be known at compile time
    sA = cuda.shared.array(shape=(TPB, TPB), dtype=float32)
    sB = cuda.shared.array(shape=(TPB, TPB), dtype=float32)

    x, y = cuda.grid(2)
    
    
    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    bpg = cuda.gridDim.x    # blocks per grid

    if x >= C.shape[0] and y >= C.shape[1]:
        # Quit if (x, y) is outside of valid C boundary
        return

    # Each thread computes one element in the result matrix.
    # The dot product is chunked into dot products of TPB-long vectors.
    tmp = 0.
    for i in range(bpg):
        # Preload data into shared memory
        sA[tx, ty] = A[x, ty + i * TPB]
        sB[tx, ty] = B[tx + i * TPB, y]

        # Wait until all threads finish preloading
        cuda.syncthreads()

        # Computes partial product on the shared memory
        for j in range(TPB):
            tmp += sA[tx, j] * sB[j, ty]

        # Wait until all threads finish computing
        cuda.syncthreads()

    C[x, y] = tmp

In [3]:
a = np.arange(0,10000*10000*0.01,.01,dtype='f').reshape((10000,10000))
b = np.arange(0,10000*10000*0.01,.01,dtype='f').reshape((10000,10000))
out = np.empty_like(a)

In [4]:
#a_device = cuda.to_device(a)
#out_device = cuda.device_array_like(a)

cuda.synchronize()
start=time.time()

fast_matmul(a, b, out)
cuda.synchronize()
#out =  out_device.copy_to_host()
end=time.time()

print(end - start)
#fsjkofreggg342432 reg

1.1768925189971924


In [90]:
#  %timeit
%time out=a@a

Wall time: 9.3 s


In [1]:
for i in range(1,5):
    print(i)

1
2
3
4


In [3]:
TILE_DIM = 32
BLOCK_ROWS = 8

@cuda.jit
def transpose(a_in, a_out):
    x = cuda.blockIdx.x * TILE_DIM + cuda.threadIdx.x
    y = cuda.blockIdx.y * TILE_DIM + cuda.threadIdx.y

    for j in range(0, TILE_DIM, BLOCK_ROWS):
        a_out[x, y + j] = a_in[y + j, x]

In [5]:

size = 1024
a_in = cuda.to_device(np.arange(size*size, dtype=np.int32).reshape((size, size)))
a_out = cuda.device_array_like(a_in)

print(a_in.copy_to_host())

[[      0       1       2 ...    1021    1022    1023]
 [   1024    1025    1026 ...    2045    2046    2047]
 [   2048    2049    2050 ...    3069    3070    3071]
 ...
 [1045504 1045505 1045506 ... 1046525 1046526 1046527]
 [1046528 1046529 1046530 ... 1047549 1047550 1047551]
 [1047552 1047553 1047554 ... 1048573 1048574 1048575]]


In [57]:
start=time.time()

grid_shape = (int(size/TILE_DIM), int(size/TILE_DIM))
transpose[grid_shape,(TILE_DIM, BLOCK_ROWS)](a_in, a_out); 
cuda.synchronize()

end=time.time()
print(end - start)

0.2204599380493164


In [58]:

import numba.types

TILE_DIM_PADDED = TILE_DIM   # Read Mark Harris' blog post to find out why this improves performance!

@cuda.jit
def tile_transpose(a_in, a_out):
    # THIS CODE ASSUMES IT IS RUNNING WITH A BLOCK DIMENSION OF (TILE_SIZE x TILE_SIZE)
    # AND INPUT IS A MULTIPLE OF TILE_SIZE DIMENSIONS
    tile = cuda.shared.array((TILE_DIM, TILE_DIM_PADDED), numba.types.int32)

    x = cuda.blockIdx.x * TILE_DIM + cuda.threadIdx.x
    y = cuda.blockIdx.y * TILE_DIM + cuda.threadIdx.y
    
    for j in range(0, TILE_DIM, BLOCK_ROWS):
        tile[cuda.threadIdx.y + j, cuda.threadIdx.x] = a_in[y + j, x] # transpose tile into shared memory

    cuda.syncthreads()  # wait for all threads in the block to finish updating shared memory

    #Compute transposed offsets
    x = cuda.blockIdx.y * TILE_DIM + cuda.threadIdx.x
    y = cuda.blockIdx.x * TILE_DIM + cuda.threadIdx.y

    for j in range(0, TILE_DIM, BLOCK_ROWS):
        a_out[y + j, x] = tile[cuda.threadIdx.x, cuda.threadIdx.y + j];

In [23]:

a_out = cuda.device_array_like(a_in) # replace with new array

start=time.time()

tile_transpose[grid_shape,(TILE_DIM, BLOCK_ROWS)](a_in, a_out); 
cuda.synchronize()
#out=a_out.copy_to_host()

end=time.time()
print(end - start)

5.98 ms ± 268 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
[[0.0000e+00 1.0000e+01 2.0000e+01 ... 1.0455e+04 1.0465e+04 1.0475e+04]
 [0.0000e+00 1.0000e+01 2.0000e+01 ... 1.0455e+04 1.0465e+04 1.0475e+04]
 [0.0000e+00 1.0000e+01 2.0000e+01 ... 1.0455e+04 1.0465e+04 1.0475e+04]
 ...
 [1.0000e+01 2.0000e+01 3.0000e+01 ... 1.0465e+04 1.0475e+04 1.0485e+04]
 [1.0000e+01 2.0000e+01 3.0000e+01 ... 1.0465e+04 1.0475e+04 1.0485e+04]
 [1.0000e+01 2.0000e+01 3.0000e+01 ... 1.0465e+04 1.0475e+04 1.0485e+04]]


In [1]:
!jt -l

Available Themes: 
   chesterish
   grade3
   gruvboxd
   gruvboxl
   monokai
   oceans16
   onedork
   solarizedd
   solarizedl
